<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    scripts = p / "work" / "scripts"
    if (scripts / "warehouse_frame.py").exists():
        sys.path.insert(0, str(scripts))
        break
else:
    raise FileNotFoundError("work/scripts/warehouse_frame.py not found")

from warehouse_frame import load_notebook_frame

df = load_notebook_frame()

Hugging Face warehouse: 79,576 pages, 26 clients
Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
Declining rate: 0.557


In [12]:
# Create the target label
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Target created successfully")
print(f"Declining rate: {df['is_declining'].mean():.3f}")

Target created successfully
Declining rate: 0.557


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Research Paper Claim Audit & Methodology Questions

**Finding #1 Audit (Content Refresh Performance Claims)**: The FlyRank research paper likely claims that ML-driven content refresh prioritization significantly outperforms manual selection methods for identifying declining pages.

*Methodology Question*: Where does the label come from? If the paper uses current trend_direction as the label for "needs refresh," this is a proxy label rather than a true future outcome. A more rigorous validation would use forward-looking labels like "pages that actually recovered after refresh" vs "pages that continued declining." Does the validation design account for this proxy limitation?

**Finding #2 Audit (Position vs CTR Relationships)**: The paper likely reports strong correlations between search position and click-through rate as justification for refresh prioritization.

*Methodology Question*: Does the validation design control for query intent and SERP layout differences? Transactional queries with shopping ads vs informational queries with featured snippets have different baseline CTR expectations. If the analysis aggregates across intent types without segmentation, the position-CTR relationship may be confounded by intent rather than pure ranking effects.

In [13]:
print("Research paper analysis completed - methodology questions logged above")

Research paper analysis completed - methodology questions logged above


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Random split vs client-holdout
Week-5 already used GroupKFold. This cell still trains a random 80/20 split next to the grouped split so the gap is visible. The number we report is the client-holdout fold 1 Precision@50 in `canonical_metrics.json` (0.280 on the warehouse). The mean across five grouped folds can differ from that first fold.

In [14]:
# Use the five core features from the data contract
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']
X = df[features].fillna(0)
y = df['is_declining']
groups = df['client_id']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("=== RANDOM SPLIT (leak check, not the reported number) ===")
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_rand = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_rand.fit(X_train_rand, y_train_rand)
rand_p50 = precision_at_k(rf_rand.predict_proba(X_test_rand)[:, 1], y_test_rand.values, 50)

print(f"Random split Precision@50: {rand_p50:.3f}")

print("\n=== CLIENT-HOLDOUT (reported design) ===")
group_kfold = GroupKFold(n_splits=5)
client_holdout_scores = []

for fold, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups=groups), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf_cv = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf_cv.fit(X_train, y_train)
    cv_p50 = precision_at_k(rf_cv.predict_proba(X_test)[:, 1], y_test.values, 50)
    client_holdout_scores.append(cv_p50)

    train_clients = df.iloc[train_idx]['client_id'].nunique()
    test_clients = df.iloc[test_idx]['client_id'].nunique()
    extra = "  <- reported fold" if fold == 1 else ""
    print(f"Fold {fold}: Precision@50 = {cv_p50:.3f} (Train: {train_clients} clients, Test: {test_clients} clients){extra}")

print(f"\nFirst-fold Precision@50 (matches the report): {client_holdout_scores[0]:.3f}")
print(f"Five-fold mean Precision@50: {np.mean(client_holdout_scores):.3f} ± {np.std(client_holdout_scores):.3f}")

comparison = pd.DataFrame({
    'Validation Strategy': ['Random 80/20', 'Client-holdout fold 1', 'Client-holdout 5-fold mean'],
    'Precision@50': [rand_p50, client_holdout_scores[0], np.mean(client_holdout_scores)],
})

print("\n=== RANDOM vs GROUPED ===")
print(comparison.to_string(index=False))

gap = rand_p50 - client_holdout_scores[0]
print(f"\nRandom minus fold-1 holdout: {gap:.3f}")
print("A gap here means some of the random-split score was client-specific.")

=== RANDOM SPLIT (leak check, not the reported number) ===


Random split Precision@50: 0.960

=== CLIENT-HOLDOUT (reported design) ===
Fold 1: Precision@50 = 0.280 (Train: 25 clients, Test: 1 clients)  <- reported fold
Fold 2: Precision@50 = 0.880 (Train: 25 clients, Test: 1 clients)
Fold 3: Precision@50 = 0.600 (Train: 17 clients, Test: 9 clients)
Fold 4: Precision@50 = 0.440 (Train: 15 clients, Test: 11 clients)
Fold 5: Precision@50 = 0.520 (Train: 22 clients, Test: 4 clients)

First-fold Precision@50 (matches the report): 0.280
Five-fold mean Precision@50: 0.544 ± 0.199

=== RANDOM vs GROUPED ===
       Validation Strategy  Precision@50
              Random 80/20         0.960
     Client-holdout fold 1         0.280
Client-holdout 5-fold mean         0.544

Random minus fold-1 holdout: 0.680
A gap here means some of the random-split score was client-specific.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Comprehensive Data Space Leakage Audit
I need to verify that no label-derived features or future information leaks into my model. The leakage taxonomy includes: (1) Label-derived features, (2) Future/overlapping windows, (3) Decision-derived features.

In [15]:
print("=== LEAKAGE AUDIT ===")

print("\n1. LABEL-DERIVED FEATURES CHECK:")
label_features = ["trend_pct"]
found_label_features = [feat for feat in label_features if feat in features]
if found_label_features:
    print(f"   WARNING: Found label-derived features: {found_label_features}")
else:
    print("   No label-derived features in model (trend_pct excluded)")

print("\n2. FEATURE-TARGET CORRELATION CHECK:")
for feat in features:
    corr = df[feat].corr(df["is_declining"])
    status = "HIGH" if abs(corr) > 0.7 else "OK"
    print(f"   {feat}: {corr:.3f} {status}")

print("\n3. FUTURE WINDOW OVERLAP CHECK:")
print("   Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.")
print("   Features sit before the label window. This is a future outcome, not a same-window proxy.")

print("\n4. PRODUCT DECISION FLAGS CHECK:")
product_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
found_flags = [flag for flag in product_flags if flag in df.columns]
if found_flags:
    print(f"   Found product flags in dataset: {found_flags}")
    print("   These are excluded from features")
else:
    print("   No product decision flags in dataset")

print("\n5. DELIBERATE LEAKAGE TEST:")
X_leaky_train = X_train_rand.copy()
X_leaky_test = X_test_rand.copy()
X_leaky_train["trend_pct"] = df.loc[X_train_rand.index, "trend_pct"].fillna(0)
X_leaky_test["trend_pct"] = df.loc[X_test_rand.index, "trend_pct"].fillna(0)

rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
rf_leaky.fit(X_leaky_train, y_train_rand)
leaky_p50 = precision_at_k(rf_leaky.predict_proba(X_leaky_test)[:, 1], y_test_rand.values, 50)

rf_honest = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
rf_honest.fit(X_train_rand, y_train_rand)
honest_p50 = precision_at_k(rf_honest.predict_proba(X_test_rand)[:, 1], y_test_rand.values, 50)

print(f"   With trend_pct (leaky): {leaky_p50:.3f}")
print(f"   Without trend_pct (honest): {honest_p50:.3f}")
print(f"   Gap: {(leaky_p50 - honest_p50):.3f}")
print("   If the leaky run jumps toward 1.0, trend_pct must stay out of the model.")

print("\n=== LEAKAGE AUDIT SUMMARY ===")
print("No label-derived features in final model")
print("No product decision flags as features")
print("No future-window features (Jan-Feb before Apr-vs-Mar)")
print("Limitation: a future drop label is not a claim that a refresh will recover traffic")

=== LEAKAGE AUDIT ===

1. LABEL-DERIVED FEATURES CHECK:
   No label-derived features in model (trend_pct excluded)

2. FEATURE-TARGET CORRELATION CHECK:
   content_age_days: -0.084 OK
   days_since_last_update: 0.014 OK
   impressions_90d: -0.066 OK
   ctr: -0.099 OK
   avg_position: -0.054 OK

3. FUTURE WINDOW OVERLAP CHECK:
   Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
   Features sit before the label window. This is a future outcome, not a same-window proxy.

4. PRODUCT DECISION FLAGS CHECK:
   No product decision flags in dataset

5. DELIBERATE LEAKAGE TEST:
   With trend_pct (leaky): 1.000
   Without trend_pct (honest): 0.920
   Gap: 0.080
   If the leaky run jumps toward 1.0, trend_pct must stay out of the model.

=== LEAKAGE AUDIT SUMMARY ===
No label-derived features in final model
No product decision flags as features
No future-window features (Jan-Feb before Apr-vs-Mar)
Limitation: a future drop label is not a claim that a refresh will recover traffic


## 4. Claim rewrite

*Rewrite any of your own claims that go further than the evidence, using safe claim language: observed, measured, directional, decision-support.*

### Claim Rewrite Using Safe Language

**Original Strong Claim**: "My model accurately predicts which pages need content refresh and will improve their search performance."

**Rewritten Honest Claim**: "On the Hugging Face warehouse, Jan–Feb search signals rank pages for a measured Apr-vs-Mar impression drop. Client-holdout fold 1 Precision@50 is 0.280 for the Random Forest and 0.640 for the fair hand-written rule. The forest does not beat the rule on this split. That is directional decision-support for who to review first, not a claim that a rewrite will move rank."

**Key Language Changes**:
- "accurately predicts" → "ranks pages by observed Jan–Feb signals"
- "will improve their search performance" → "decision-support for who to look at first"
- Dropped "beats the baseline" — measured forest 0.280 vs rule 0.640
- Label is a future outcome (Apr vs Mar), not a current-state proxy

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.